# Q-Trust — GPU Quantum Threat Demo

Executes Shor's algorithm via `qiskit-aer(-gpu)` when a simulator is available (with an honestly-labeled classical fallback otherwise), then derives RSA threat estimates from published hardware roadmaps.

Generated from `notebooks/02_quantum_threat_gpu.py` — keep the script authoritative and regenerate this notebook after edits.

Quantum threat demonstration — GPU-aware Shor simulation + RSA threat estimates.

Run as a script or in Jupyter:
    python notebooks/02_quantum_threat_gpu.py

Uses qiskit-aer(-gpu) when available; otherwise falls back to an
honestly-labeled classical Pollard's rho for small N. All quantum resource
estimates come from published roadmaps via
qtrust_planner.quantum_estimator.QuantumThreatEstimator.

In [1]:
import sys
import time
from pathlib import Path

# Make qtrust_planner importable regardless of the working directory
# (works as a script, from the repo root, and inside Jupyter where __file__
# is undefined).
_bootstrap = [Path.cwd() / "planner"]
try:
    _bootstrap.append(Path(__file__).resolve().parent.parent / "planner")
except NameError:
    pass  # executed inside a notebook — no __file__
for _p in _bootstrap:
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

# Check for GPU simulator
gpu_available = False
sim_error = None
try:
    from qiskit_aer import AerSimulator  # noqa: F401

    test_sim = AerSimulator(method="statevector", device="GPU")
    gpu_available = True
    print("✓ GPU-accelerated simulator available")
except ImportError:
    sim_error = "qiskit-aer not installed (pip install qiskit-aer)"
except Exception as exc:  # GPU driver/CUDA issues
    sim_error = str(exc)

if sim_error:
    print(f"⚠ quantum simulator unavailable: {sim_error}")
    print("  factor() will use the classical fallback with honest method labels.")

from qtrust_planner.quantum_estimator import QuantumThreatEstimator  # noqa: E402 - notebook cell ordering

estimator = QuantumThreatEstimator()

✓ GPU-accelerated simulator available


## Factoring Demo

In [2]:
numbers = [15, 21, 35, 77]
results = {}
for N in numbers:
    print(f"\nFactoring N={N}...")
    start = time.time()
    result = estimator.factor(N, use_gpu=gpu_available)
    elapsed = time.time() - start
    results[N] = result
    status = "✓" if result.success else "✗"
    print(
        f"  {status} factors={result.factors} "
        f"method={result.method!r} circuit_qubits={result.quantum_circuit_qubits} "
        f"time={elapsed:.2f}s"
    )


Factoring N=15...
  ✓ factors=[3, 5] method='classical gcd shortcut' circuit_qubits=11 time=0.00s

Factoring N=21...
Quantum order-finding on GPU failed (Simulation device "GPU" is not supported on this system); trying next device.


Quantum order-finding on GPU failed (Simulation device "GPU" is not supported on this system); trying next device.


  ✓ factors=[3, 7] method='shor (cpu aer simulation)' circuit_qubits=13 time=0.91s

Factoring N=35...
Quantum order-finding on GPU failed (Simulation device "GPU" is not supported on this system); trying next device.


  ✓ factors=[5, 7] method='shor (cpu aer simulation)' circuit_qubits=15 time=0.78s

Factoring N=77...


Quantum order-finding on GPU failed (Simulation device "GPU" is not supported on this system); trying next device.


Quantum order-finding on GPU failed (Simulation device "GPU" is not supported on this system); trying next device.


Quantum order-finding on GPU failed (Simulation device "GPU" is not supported on this system); trying next device.


  ✓ factors=[7, 11] method='classical gcd shortcut' circuit_qubits=17 time=8.19s


## Rsa Threat Estimates

In [3]:
print("\n=== Quantum Threat Estimates (Gidney & Ekerå 2019 + hardware roadmaps) ===")
report = estimator.generate_threat_report([1024, 2048, 3072, 4096])
for key, est in report["key_sizes"].items():
    year = est["estimated_breakable_year"] or "after 2033"
    print(
        f"{key:>8}: {est['logical_qubits_needed']:>6,} logical qubits | "
        f"{est['physical_qubits_needed']:>12,} physical | breakable ~{year}"
    )

out_path = "quantum_threat_report.json"
estimator.save_report(out_path)
print(f"\nFull report saved to {out_path}")


=== Quantum Threat Estimates (Gidney & Ekerå 2019 + hardware roadmaps) ===
RSA-1024:  2,051 logical qubits |    2,051,000 physical | breakable ~after 2033
RSA-2048:  4,099 logical qubits |    4,099,000 physical | breakable ~after 2033
RSA-3072:  6,147 logical qubits |    6,147,000 physical | breakable ~after 2033
RSA-4096:  8,195 logical qubits |    8,195,000 physical | breakable ~after 2033
Report saved to quantum_threat_report.json

Full report saved to quantum_threat_report.json
